In [1]:
import river
import copy
import math
import typing
import time
import os
import numpy as np
import seaborn as sns
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

from river import stream
from river import base, utils
from abc import ABCMeta
from collections import defaultdict, deque
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import MinMaxScaler

In [6]:
from __future__ import annotations

import copy
import math
from abc import ABCMeta
from collections import defaultdict, deque

from river import base, utils


class DenStream(base.Clusterer):
    r"""DenStream

    DenStream [^1] is a clustering algorithm for evolving data streams.
    DenStream can discover clusters with arbitrary shape and is robust against
    noise (outliers).

    "Dense" micro-clusters (named core-micro-clusters) summarise the clusters
    of arbitrary shape. A pruning strategy based on the concepts of potential
    and outlier micro-clusters guarantees the precision of the weights of the
    micro-clusters with limited memory.

    The algorithm is divided into two parts:

    **Online micro-cluster maintenance (learning)**

    For a new point `p`:

    * Try to merge `p` into either the nearest `p-micro-cluster` (potential),
    `o-micro-cluster` (outlier), or create a new `o-micro-cluster` and insert it
    into the outlier buffer.

    * For each `T_p` iterations, consider the weights of all potential and
    outlier micro-clusters. If their weights are smaller than a certain
    threshold (different for each type of micro-clusters), the micro-cluster is
    deleted.

    **Offline generation of clusters on-demand (clustering)**

    A variant of the DBSCAN algorithm [^2] is used, such that all
    density-connected p-micro-clusters determine the final clusters. Moreover,
    in order for the algorithm to always be able to generate clusters, a certain
    number of points must be passed through the algorithm with a suitable streaming
    speed (number of points passed through within a unit time), indicated by
    `n_samples_init` and `stream_speed`.

    Parameters
    ----------
    decaying_factor
        Parameter that controls the importance of historical data to current cluster.
        Note that `decaying_factor` has to be different from `0`.

    beta
        Parameter to determine the threshold of outlier relative to core micro-clusters.
        The value of `beta` must be within the range `(0,1]`.

    mu
        Parameter to determine the threshold of outliers relative to core micro-cluster.
        As `beta * mu` must be greater than 1, `mu` must be within the range `(1/beta, inf)`.

    epsilon
        Defines the epsilon neighborhood

    n_samples_init
        Number of points to to initiqalize the online process

    stream_speed
        Number of points arrived in unit time

    Attributes
    ----------
    n_clusters
        Number of clusters generated by the algorithm.

    clusters
        A set of final clusters of type `MicroCluster`, which means that these cluster include all
        the required information, including number of points, creation time, weight, (weighted)
        linear sum, (weighted) square sum, center and radius.

    p_micro_clusters
        The potential core-icro-clusters that are generated by the algorithm. When a generate
        cluster request arrives, these p-micro-clusters will go through a variant of the DBSCAN
        algorithm to determine the final clusters.

    o_micro_clusters
        The outlier micro-clusters.

    References
    ----------
    [^1]: Feng et al (2006, pp 328-339). Density-Based Clustering over an Evolving Data Stream with
          Noise. In Proceedings of the Sixth SIAM International Conference on Data Mining,
          April 20–22, 2006, Bethesda, MD, USA.
    [^2]: Ester et al (1996). A Density-Based Algorithm for Discovering Clusters in Large Spatial
          Databases with Noise. In KDD-96 Proceedings, AAAI.

    Examples
    --------

    The following example uses the default parameters of the algorithm to test its functionality.
    The set of evolving points `X` are designed so that clusters are easily identifiable.

    >>> from river import cluster
    >>> from river import stream

    >>> X = [
    ...     [-1, -0.5], [-1, -0.625], [-1, -0.75], [-1, -1], [-1, -1.125],
    ...     [-1, -1.25], [-1.5, -0.5], [-1.5, -0.625], [-1.5, -0.75], [-1.5, -1],
    ...     [-1.5, -1.125], [-1.5, -1.25], [1, 1.5], [1, 1.75], [1, 2],
    ...     [4, 1.25], [4, 1.5], [4, 2.25], [4, 2.5], [4, 3],
    ...     [4, 3.25], [4, 3.5], [4, 3.75], [4, 4],
    ... ]

    >>> denstream = cluster.DenStream(decaying_factor=0.01,
    ...                               beta=0.5,
    ...                               mu=2.5,
    ...                               epsilon=0.5,
    ...                               n_samples_init=10)

    >>> for x, _ in stream.iter_array(X):
    ...     denstream = denstream.learn_one(x)

    >>> denstream.predict_one({0: -1, 1: -2})
    1

    >>> denstream.predict_one({0:5, 1:4})
    2

    >>> denstream.predict_one({0:1, 1:1})
    0

    >>> denstream.n_clusters
    3

    """

    class BufferItem:
        def __init__(self, x, timestamp, covered):
            self.x = x
            self.timestamp = (timestamp,)
            self.covered = covered

    def __init__(
        self,
        decaying_factor: float = 0.25,
        beta: float = 0.75,
        mu: float = 2,
        epsilon: float = 0.02,
        n_samples_init: int = 1000,
        stream_speed: int = 100,
    ):
        super().__init__()
        self.timestamp = -1
        self.initialized = False
        self.decaying_factor = decaying_factor
        self.beta = beta
        self.mu = mu
        self.epsilon = epsilon
        self.n_samples_init = n_samples_init
        self.stream_speed = stream_speed

        # number of clusters generated by applying the variant of DBSCAN algorithm
        # on p-micro-cluster centers and their centers
        self.n_clusters = 0
        self.clusters: dict[int, DenStreamMicroCluster] = {}
        self.p_micro_clusters: dict[int, DenStreamMicroCluster] = {}
        self.o_micro_clusters: dict[int, DenStreamMicroCluster] = {}

        self._time_period = math.ceil(
            (1 / self.decaying_factor) * math.log((self.mu * self.beta) / (self.mu * self.beta - 1))
        )
        self._init_buffer: deque[dict] = deque()
        self._n_samples_seen = 0

        # check that the value of beta is within the range (0,1]
        if not (0 < self.beta <= 1):
            raise ValueError(
                f"The value of `beta` (currently {self.beta}) must be within the range (0,1]."
            )

    @property
    def centers(self):
        return {k: cluster.calc_center(self.timestamp) for k, cluster in self.clusters.items()}

    @staticmethod
    def _distance(point_a, point_b):
        return math.sqrt(utils.math.minkowski_distance(point_a, point_b, 2))

    def _get_closest_cluster_key(self, point, clusters):
        min_distance = math.inf
        key = -1
        for k, cluster in clusters.items():
            center = cluster.calc_center(self.timestamp)
            distance = self._distance(center, point)
            if distance < min_distance:
                min_distance = distance
                key = k
        return key

    def _merge(self, point):
        # initiate merged status
        merged_status = False

        if len(self.p_micro_clusters) != 0:
            # try to merge p into its nearest p-micro-cluster c_p
            closest_pmc_key = self._get_closest_cluster_key(point, self.p_micro_clusters)
            updated_pmc     = copy.copy(self.p_micro_clusters[closest_pmc_key])
            updated_pmc.insert(point, self.timestamp)
            
            if updated_pmc.calc_radius(self.timestamp) <= self.epsilon:
                # keep updated p-micro-cluster
                self.p_micro_clusters[closest_pmc_key] = updated_pmc
                merged_status = True

        if not merged_status and len(self.o_micro_clusters) != 0:
            closest_omc_key = self._get_closest_cluster_key(point, self.o_micro_clusters)
            updated_omc     = copy.copy(self.o_micro_clusters[closest_omc_key])
            updated_omc.insert(point, self.timestamp)
            
            if updated_omc.calc_radius(self.timestamp) <= self.epsilon:
                # keep updated o-micro-cluster
                if updated_omc.calc_weight(self.timestamp) > self.mu * self.beta:
                    # it has grown into a p-micro-cluster
                    del self.o_micro_clusters[closest_omc_key]
                    
                    if self.p_micro_clusters.get(len(self.p_micro_clusters)) is not None:
                        print("ACONTECEU O PROBLEMA!!")
                    
                    self.p_micro_clusters[len(self.p_micro_clusters)] = updated_omc
                else:
                    self.o_micro_clusters[closest_omc_key] = updated_omc
            else:
                # create a new o-micro-cluster by p and add it to o_micro_clusters
                mc_from_p = DenStreamMicroCluster(
                    x=point,
                    timestamp=self.timestamp,
                    decaying_factor=self.decaying_factor,
                )
                self.o_micro_clusters[len(self.o_micro_clusters)] = mc_from_p
            merged_status = True

        # when p is not merged and o-micro-cluster set is empty
        if not merged_status and len(self.o_micro_clusters) == 0:
            mc_from_p = DenStreamMicroCluster(
                x=point, timestamp=self.timestamp, decaying_factor=self.decaying_factor
            )
            self.o_micro_clusters = {0: mc_from_p}
            merged_status = True

    def _is_directly_density_reachable(self, c_p, c_q):
        if c_p.calc_weight(self.timestamp) > self.mu and c_q.calc_weight(self.timestamp) > self.mu:
            # check distance of two clusters and compare with 2*epsilon
            c_p_center = c_p.calc_center(self.timestamp)
            c_q_center = c_q.calc_center(self.timestamp)
            distance = self._distance(c_p_center, c_q_center)
            if distance < 2 * self.epsilon and distance <= c_p.calc_radius(
                self.timestamp
            ) + c_q.calc_radius(self.timestamp):
                return True
        return False

    def _query_neighbor(self, cluster):
        neighbors = deque()
        # scan all clusters within self.p_micro_clusters
        for pmc in self.p_micro_clusters.values():
            # check density reachable and that the cluster itself does not appear in neighbors
            if cluster != pmc and self._is_directly_density_reachable(cluster, pmc):
                neighbors.append(pmc)
        return neighbors

    @staticmethod
    def _generate_clusters_for_labels(cluster_labels):
        # initiate the dictionary for final clusters
        clusters = {}

        # group clusters per label
        mcs_per_label = defaultdict(deque)
        for mc, label in cluster_labels.items():
            mcs_per_label[label].append(mc)

        # generate set of clusters with the same label
        for label, micro_clusters in mcs_per_label.items():
            # merge clusters with the same label into a big cluster
            cluster = copy.copy(micro_clusters[0])
            for mc in range(1, len(micro_clusters)):
                cluster.merge(micro_clusters[mc])

            clusters[label] = cluster

        return len(clusters), clusters

    def _expand_cluster(self, mc, neighborhood):
        for idx in neighborhood:
            item = self._init_buffer[idx]
            if not item.covered:
                item.covered = True
                mc.insert(item.x, self.timestamp)

    def _get_neighborhood_ids(self, item):
        neighborhood_ids = deque()
        for idx, other in enumerate(self._init_buffer):
            if not other.covered:
                if self._distance(item.x, other.x) < self.epsilon:
                    neighborhood_ids.append(idx)
        return neighborhood_ids

    def _initial_dbscan(self):
        for item in self._init_buffer:
            if not item.covered:
                item.covered = True
                neighborhood = self._get_neighborhood_ids(item)
                if len(neighborhood) > self.mu:
                    mc = DenStreamMicroCluster(
                        x=item.x,
                        timestamp=self.timestamp,
                        decaying_factor=self.decaying_factor,
                    )
                    self._expand_cluster(mc, neighborhood)
                    self.p_micro_clusters.update({len(self.p_micro_clusters): mc})
                else:
                    item.covered = False

    def learn_one(self, x, sample_weight=None):
        self._n_samples_seen += 1
        # control the stream speed
        if self._n_samples_seen % self.stream_speed == 0:
            self.timestamp += 1

        # Initialization
        if not self.initialized:
            self._init_buffer.append(self.BufferItem(x, self.timestamp, False))
            if len(self._init_buffer) == self.n_samples_init:
                self._initial_dbscan()
                print("DBs: ", len(self.p_micro_clusters))
                self.initialized = True
                del self._init_buffer
            return self

        # Merge
        self._merge(x)

        # Periodic cluster removal
        if self.timestamp > 0 and self.timestamp % self._time_period == 0:
            for i, p_micro_cluster_i in list(self.p_micro_clusters.items()):
                if p_micro_cluster_i.calc_weight(self.timestamp) < self.mu * self.beta:
                    # c_p became an outlier and should be deleted
                    del self.p_micro_clusters[i]
            for j, o_micro_cluster_j in list(self.o_micro_clusters.items()):
                # calculate xi
                xi = (2** (-self.decaying_factor * (self.timestamp - o_micro_cluster_j.creation_time + self._time_period)) - 1) / (2 ** (-self.decaying_factor * self._time_period) - 1)
                if o_micro_cluster_j.calc_weight(self.timestamp) < xi:
                    # c_o might not grow into a p-micro-cluster, we can safely delete it
                    self.o_micro_clusters.pop(j)
        return self

    def predict_one(self, x, sample_weight=None):
        # This function handles the case when a clustering request arrives.
        # implementation of the DBSCAN algorithm proposed by Ester et al.
        if not self.initialized:
            # The model is not ready
            return 0

        # cluster counter; in this algorithm cluster labels start with 0
        c = -1
        # initiate labels of p-micro-clusters to None
        labels = {pmc: None for pmc in self.p_micro_clusters.values()}

        for pmc in self.p_micro_clusters.values():
            # previously processed in inner loop
            if labels[pmc] is not None:
                continue
            # next cluster label
            c += 1
            labels[pmc] = c
            # neighbors to expand
            seed_queue = self._query_neighbor(pmc)
            # process every point in seed set
            while seed_queue:
                # check previously proceeded points
                if labels[seed_queue[0]] is not None:
                    seed_queue.popleft()
                    continue
                if seed_queue:
                    labels[seed_queue[0]] = c
                    # find neighbors of neighbors
                    neighbor_neighbors = self._query_neighbor(seed_queue[0])
                    # add new neighbors to seed set
                    for neighbor_neighbor in neighbor_neighbors:
                        if labels[neighbor_neighbor] is not None:
                            seed_queue.append(neighbor_neighbor)

        self.n_clusters, self.clusters = self._generate_clusters_for_labels(labels)

        return self._get_closest_cluster_key(x, self.clusters)


class DenStreamMicroCluster(metaclass=ABCMeta):
    """DenStream Micro-cluster class"""

    def __init__(self, x, timestamp, decaying_factor):
        self.x = x
        self.last_edit_time  = timestamp
        self.creation_time   = timestamp
        self.decaying_factor = decaying_factor

        self.N           = 1
        self.linear_sum  = x
        self.squared_sum = {i: (x_val * x_val) for i, x_val in x.items()}

    def calc_norm_cf1_cf2(self, fading_factor):
        # |CF1| and |CF2| in the paper
        sum_of_squares_cf1 = 0
        sum_of_squares_cf2 = 0
        
        for key in self.linear_sum.keys():
            val_ls = self.linear_sum[key]
            val_ss = self.squared_sum[key]
            sum_of_squares_cf1 += fading_factor * val_ls * fading_factor * val_ls
            sum_of_squares_cf2 += fading_factor * val_ss * fading_factor * val_ss
            
        # return |CF1| and |CF2|
        return math.sqrt(sum_of_squares_cf1), math.sqrt(sum_of_squares_cf2)

    def calc_weight(self, timestamp):
        return self._weight(self.fading_function(timestamp - self.last_edit_time))

    def _weight(self, fading_factor):
        return self.N * fading_factor

    def calc_center(self, timestamp):
        ff     = self.fading_function(timestamp - self.last_edit_time)
        weight = self._weight(ff)
        center = {key: (ff * val) / weight for key, val in self.linear_sum.items()}
        
        return center

    def calc_radius(self, timestamp):
        ff     = self.fading_function(timestamp - self.last_edit_time)
        weight = self._weight(ff)
        
        norm_cf1, norm_cf2 = self.calc_norm_cf1_cf2(ff)
        
        diff   = (norm_cf2 / weight) - ((norm_cf1 / weight) ** 2)
        radius = math.sqrt(diff) if diff > 0 else 0
        
        return radius

    def insert(self, x, timestamp):
        self.N += 1
        self.last_edit_time = timestamp
        
        for key, val in x.items():
            try:
                self.linear_sum[key] += val
                self.squared_sum[key] += val * val
            except KeyError:
                self.linear_sum[key] = val
                self.squared_sum[key] = val * val

    def merge(self, cluster):
        self.N += cluster.N
        
        for key in cluster.linear_sum.keys():
            try:
                self.linear_sum[key] += cluster.linear_sum[key]
                self.squared_sum[key] += cluster.squared_sum[key]
            except KeyError:
                self.linear_sum[key] = cluster.linear_sum[key]
                self.squared_sum[key] = cluster.squared_sum[key]
            
        if self.last_edit_time < cluster.creation_time:
            self.last_edit_time = cluster.creation_time

    def fading_function(self, time):
        return 2 ** (-self.decaying_factor * time)

In [7]:
data = pd.read_csv("datasets/dataset_38k_x_y.csv", sep=',')

scaler = MinMaxScaler()

scaler.fit(data)

data = pd.DataFrame(data=scaler.transform(data), columns=['x', 'y'])

data = data.to_numpy()
len(data)

38600

In [8]:
#38k- 0.0048 = 15%
#     0.0069 = 10%
#     0.0115 = 5%

#43k- 0.0036 = 15%
#     0.0054 = 10%
#     0.0099 = 5%

#49k- 0.0043 = 15%
#     0.0061 = 10%
#     0.0105 = 5%


#0.027
denstream = DenStream(200, mu=2, n_samples_init=8000, epsilon=0.003)#  0 < percent <= 1.0

In [9]:
for x, _ in stream.iter_array(data):
    denstream = denstream.learn_one(x)

DBs:  773
Extent----
LS:  {0: 0.7470236960986518, 1: 3.4762836246037154}
SS:  {0: 0.09307279067686786, 1: 2.0140949566584636}
Extent----
LS:  {0: 0.5965237545524331, 1: 3.1381724571703202}
SS:  {0: 0.07119613257898345, 1: 1.969626727809843}
Extent----
LS:  {0: 0.8883263317615343, 1: 4.193259424147612}
SS:  {0: 0.11273346846392074, 1: 2.5119584900866125}
Extent----
LS:  {0: 0.6665386814889396, 1: 3.8238150784003997}
SS:  {0: 0.07405702273031117, 1: 2.436939416103603}
Extent----
LS:  {0: 0.5466136459369458, 1: 2.9679274994554734}
SS:  {0: 0.05977041079422388, 1: 1.761722881681401}
Extent----
LS:  {0: 0.6250412175778276, 1: 2.9368564309671084}
SS:  {0: 0.07815607771239196, 1: 1.7250536898140343}
Extent----
LS:  {0: 0.6543566952663296, 1: 3.637164368439909}
SS:  {0: 0.07136743594286954, 1: 2.2048394124634716}
Extent----
LS:  {0: 0.7517934359692879, 1: 4.300245647415625}
SS:  {0: 0.08075083464760445, 1: 2.641735038602287}
Extent----
LS:  {0: 0.6584951727541778, 1: 3.7533713467059324}
SS:  {

Extent----
LS:  {0: 0.8330669507353591, 1: 4.32367182997648}
SS:  {0: 0.09922866658758268, 1: 2.670606429503027}
Extent----
LS:  {0: 0.7708720534195218, 1: 4.153041469746744}
SS:  {0: 0.08491911346824771, 1: 2.4639725805831336}
Extent----
LS:  {0: 0.7583942383421238, 1: 3.6665554124607977}
SS:  {0: 0.0958759086619168, 1: 2.2406177666536378}
Extent----
LS:  {0: 0.8858110653065762, 1: 4.277611155787152}
SS:  {0: 0.11211095645560598, 1: 2.614006888105761}
Extent----
LS:  {0: 0.7793640291175296, 1: 3.4351289341690157}
SS:  {0: 0.10124292234929455, 1: 1.966693313643538}
Extent----
LS:  {0: 0.9760602117928402, 1: 5.238734830584515}
SS:  {0: 0.10587127973540829, 1: 3.049420982136486}
Extent----
LS:  {0: 1.4202810642531103, 1: 5.878723175168202}
SS:  {0: 0.20176344844321126, 1: 3.455973492195658}
Extent----
LS:  {0: 0.8732439070842197, 1: 3.6096318189685626}
SS:  {0: 0.12710362003224762, 1: 2.1715802866669867}
Extent----
LS:  {0: 0.9838859185204889, 1: 5.446697846858001}
SS:  {0: 0.10756807133

SS:  {0: 58.58909218331321, 1: 558.8336800152704}
Extent----
LS:  {0: 289.31739939339553, 1: 914.7980248486055}
SS:  {0: 58.654788906389236, 1: 559.1638290385489}
Extent----
LS:  {0: 289.5758795431044, 1: 915.3920028753683}
SS:  {0: 58.72160089418276, 1: 559.516638934826}
Extent----
LS:  {0: 289.8028174289606, 1: 916.0010619916438}
SS:  {0: 58.773101698219655, 1: 559.8875919419443}
Extent----
LS:  {0: 290.0533160994572, 1: 916.6079804464786}
SS:  {0: 58.83585128214022, 1: 560.2559419527632}
Extent----
LS:  {0: 290.31297574140405, 1: 917.2100150752971}
SS:  {0: 58.90327441179616, 1: 560.6183876470598}
Extent----
LS:  {0: 290.54700657801305, 1: 917.8041863327074}
SS:  {0: 58.95804484428008, 1: 560.9714271301924}
Extent----
LS:  {0: 290.7961186465304, 1: 918.3960402708639}
SS:  {0: 59.020101666961075, 1: 561.3217182143037}
Extent----
LS:  {0: 291.0266236624163, 1: 918.9561786538352}
SS:  {0: 59.07323422930965, 1: 561.6354732223814}
Extent----
LS:  {0: 291.28150082082107, 1: 919.5652306184

LS:  {0: 2852.5996429321917, 1: 3477.204745962402}
SS:  {0: 1094.2269909023175, 1: 1606.0202648013628}
Extent----
LS:  {0: 2853.132173463303, 1: 3478.0437080760867}
SS:  {0: 1094.510579668883, 1: 1606.7241222295618}
Extent----
LS:  {0: 2853.7838412028154, 1: 3478.8555668438403}
SS:  {0: 1094.9352505116042, 1: 1607.38323688834}
Extent----
LS:  {0: 2854.3636635903495, 1: 3479.6364113586833}
SS:  {0: 1095.2714445126896, 1: 1607.9929550447005}
Extent----
LS:  {0: 2855.0125814931193, 1: 3480.34918070865}
SS:  {0: 1095.6925389572245, 1: 1608.5009951909522}
Extent----
LS:  {0: 2855.5902198169506, 1: 3481.126395214168}
SS:  {0: 1096.026204990383, 1: 1609.1050575785405}
Extent----
LS:  {0: 2856.2261745321393, 1: 3481.8409146486774}
SS:  {0: 1096.4306433901538, 1: 1609.6155956008322}
Extent----
LS:  {0: 2856.7721271154237, 1: 3482.5557992324693}
SS:  {0: 1096.7287076133487, 1: 1610.1266555689756}
Extent----
LS:  {0: 2857.376294656508, 1: 3483.291284341621}
SS:  {0: 1097.0937260310484, 1: 1610.66

LS:  {0: 5568.2572319916835, 1: 6332.638493497756}
SS:  {0: 2936.1636757698975, 1: 3743.7229248593253}
Extent----
LS:  {0: 5568.745634698899, 1: 6333.5450012328865}
SS:  {0: 2936.4022129743134, 1: 3744.544681133177}
Extent----
LS:  {0: 5569.213059178275, 1: 6334.44937495026}
SS:  {0: 2936.6206986182333, 1: 3745.3625729538535}
Extent----
LS:  {0: 5569.713843879344, 1: 6335.301021276285}
SS:  {0: 2936.8714839350578, 1: 3746.0878744184856}
Extent----
LS:  {0: 5570.212008274445, 1: 6336.236714396136}
SS:  {0: 2937.1196516996047, 1: 3746.9633960330243}
Extent----
LS:  {0: 5570.711553738973, 1: 6337.154577887454}
SS:  {0: 2937.369197370735, 1: 3747.8058694217175}
Extent----
LS:  {0: 5571.24097515865, 1: 6338.063941063339}
SS:  {0: 2937.649484410348, 1: 3748.6328108073735}
Extent----
LS:  {0: 5571.712783336202, 1: 6338.982465137412}
SS:  {0: 2937.8720873667535, 1: 3749.476497282024}
Extent----
LS:  {0: 5572.210130174974, 1: 6339.840168020466}
SS:  {0: 2938.1194412447903, 1: 3750.2121515176227

Extent----
LS:  {0: 8223.259512980378, 1: 10013.749958640434}
SS:  {0: 4535.775233183926, 1: 6819.406964258074}
Extent----
LS:  {0: 8223.906966304086, 1: 10014.461385388162}
SS:  {0: 4536.194428990307, 1: 6819.913092275458}
Extent----
LS:  {0: 8224.549170235183, 1: 10015.23634164917}
SS:  {0: 4536.606854879423, 1: 6820.513649481933}
Extent----
LS:  {0: 8225.187010605363, 1: 10015.973489872185}
SS:  {0: 4537.013695217254, 1: 6821.057036984628}
Extent----
LS:  {0: 8225.835191922659, 1: 10016.713054029118}
SS:  {0: 4537.433834237345, 1: 6821.603992126848}
Extent----
LS:  {0: 8226.485838725283, 1: 10017.455607354002}
SS:  {0: 4537.85717549911, 1: 6822.155377567145}
Extent----
LS:  {0: 8227.115897301597, 1: 10018.202367303846}
SS:  {0: 4538.2541493086965, 1: 6822.713027989834}
Extent----
LS:  {0: 8227.740905685638, 1: 10018.930983298624}
SS:  {0: 4538.644784788818, 1: 6823.243909257679}
Extent----
LS:  {0: 8228.373211863121, 1: 10019.702555983687}
SS:  {0: 4539.044595890901, 1: 6823.8392336

SS:  {0: 7166.899850287305, 1: 8370.70481168136}
Extent----
LS:  {0: 11550.96392287554, 1: 12428.237682384504}
SS:  {0: 7167.65806400851, 1: 8371.09112508207}
Extent----
LS:  {0: 11551.842761012682, 1: 12428.868821402002}
SS:  {0: 7168.430420479806, 1: 8371.48946154148}
Extent----
LS:  {0: 11552.693923966623, 1: 12429.424960703032}
SS:  {0: 7169.154898853969, 1: 8371.79875246363}
Extent----
LS:  {0: 11553.561003265415, 1: 12430.055241964308}
SS:  {0: 7169.9067253643625, 1: 8372.196006931947}
Extent----
LS:  {0: 11554.427400436934, 1: 12430.638676295968}
SS:  {0: 7170.657369423181, 1: 8372.536402551306}
Extent----
LS:  {0: 11555.311013179024, 1: 12431.26832585126}
SS:  {0: 7171.438140901164, 1: 8372.932861113786}
Extent----
LS:  {0: 11556.186245360317, 1: 12431.789869486123}
SS:  {0: 7172.204172272336, 1: 8373.204868876852}
Extent----
LS:  {0: 11557.01972420187, 1: 12432.32463190434}
SS:  {0: 7172.8988592516525, 1: 8373.490839720787}
Extent----
LS:  {0: 11557.903439668235, 1: 12432.9115

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [ ]:
data

In [ ]:
data[:, 1]